# Geospatial Data Analysis
## Sales, User Density and Underserved Market Detection

**Dataset note:** This project uses a synthetic demonstration dataset because no source dataset was provided. Replace `regional_sales_dataset.csv` with your real dataset for final business conclusions.


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

df = pd.read_csv('regional_sales_dataset.csv')
df.head()

## 1. Data Cleaning


In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=['State','City','Postal Code','Revenue','Users','Latitude','Longitude'])
df['Revenue'] = pd.to_numeric(df['Revenue'])
df['Users'] = pd.to_numeric(df['Users'])
df['Revenue_per_User'] = (df['Revenue']/df['Users']).round(2)
df['Demand_Index'] = (df['Users']/df['Users'].max()*100).round(1)
df.describe()

## 2. Regional Aggregation


In [ ]:
regional = df.groupby('State', as_index=False).agg(Total_Revenue=('Revenue','sum'), Total_Users=('Users','sum'))
regional['Revenue_per_User'] = (regional['Total_Revenue']/regional['Total_Users']).round(2)
regional.sort_values('Total_Revenue', ascending=False)

## 3. Underserved Region Scoring
The score combines user demand with comparatively low revenue per user. This is a screening metric, not a claim of actual market coverage.

In [ ]:
df['Revenue_per_User_Rank'] = df['Revenue_per_User'].rank(pct=True, ascending=True)
df['User_Demand_Rank'] = df['Users'].rank(pct=True, ascending=True)
df['Underserved_Score'] = ((1-df['Revenue_per_User_Rank'])*0.5 + df['User_Demand_Rank']*0.5).round(3)
top3 = df.sort_values(['Underserved_Score','Users'], ascending=False).head(3)
top3[['State','City','Users','Revenue','Revenue_per_User','Underserved_Score']]

## 4. Interactive Spatial Map


In [ ]:
m = folium.Map(location=[22.5,79.0], zoom_start=5, tiles='CartoDB positron')
for _, r in df.iterrows():
    folium.CircleMarker([r.Latitude,r.Longitude], radius=6, popup=f"{r.City}, {r.State}<br>Revenue: ₹{r.Revenue:,.0f}<br>Users: {r.Users:,}<br>Score: {r.Underserved_Score:.3f}", fill=True).add_to(m)
HeatMap(df[['Latitude','Longitude','Users']].values.tolist(), radius=25, blur=18, min_opacity=.25).add_to(m)
m.save('Geospatial_Data_Analysis_Map.html')
m

## 5. Conclusion
The analysis highlights locations where demand is comparatively strong while revenue per user is comparatively low. These locations can be investigated as potential expansion markets. Before making a real investment decision, validate the result with population, competitor coverage, logistics, purchasing power and actual sales/service availability data.